# v28 — Context-DPO against recitation, on Google Colab

The same screen as `MAS_SHT_Kaggle_v28.ipynb` (rules: the `pretest_v28.py` docstring; plan: `V28_PLAN.md`), adapted to Colab:

* **One GPU.** The four steps run one after the other, FV first so that the main verdict (PRIMARY) is readable before the control arm (RW) has run: worker 0 FV, worker 1 FV, worker 0 RW, worker 1 RW. A big GPU (A100) runs the two workers of a step at the same time.
* **Everything is saved to Google Drive** (`MyDrive/MAS_SHT_v28`): the worker files, the adapters and the training checkpoints. A session that ends, for whatever reason, loses at most the last few minutes. **Run all the cells again in a new session and it continues where it stopped.**

**Before running**
1. Runtime → Change runtime type → **T4 GPU** (L4 or A100 are faster, if you have them).
2. Secrets (key icon on the left), with *Notebook access* on: `GITHUB_TOKEN` (only if the repo is private) and `HF_API_KEY`.
3. Runtime → Run all. Keep the tab open (Colab may stop a session whose tab is closed).

**How long:** on a T4, about 8-9 GPU-hours until PRIMARY and about 15 for everything, so several sessions. An A100 needs roughly 3-4 hours for everything.

When the last step has finished, the final cell prints the pre-registered reading. The two files `pretest_v28_w0.json` and `pretest_v28_w1.json` in the Drive folder are the results.


In [ ]:
# === Cell 1 — Colab setup ===========================================================
import os, subprocess, sys

# Google Drive: results, adapters and checkpoints live here and survive the session
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/MAS_SHT_v28'
os.makedirs(OUT_DIR, exist_ok=True)

# Dependencies, pinned like the Kaggle runs (the stack the stored samples came from).
# Colab's own torch is kept: reinstalling it breaks the CUDA build.
DEPS = ['transformers>=4.44.0,<4.50', 'peft>=0.12,<0.15', 'accelerate', 'bitsandbytes',
        'openai>=1.40.0', 'python-dotenv', 'tabulate']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS)
print('Deps installed.')

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

_hf = _secret('HF_API_KEY')
if _hf:
    for _k in ('HF_API_KEY', 'HUGGINGFACE_TOKEN', 'HF_TOKEN'):
        os.environ[_k] = _hf
    print('HF token loaded.')
else:
    print('Note: no HF_API_KEY secret; the Qwen models are public, so this is usually fine.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU: Runtime -> Change runtime type -> T4 GPU, then run again.')
NGPU = torch.cuda.device_count()
for _i in range(NGPU):
    _p = torch.cuda.get_device_properties(_i)
    print(f'cuda:{_i} {_p.name}  sm_{_p.major}{_p.minor}  {_p.total_memory / 2**30:.0f} GB')
    if (_p.major, _p.minor) < (7, 5):
        raise RuntimeError(f'{_p.name} is older than sm_75; 4-bit NF4 needs a T4 or newer.')
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30

REPO_DIR = '/content/llm_thesis'
BRANCH   = 'Context-DPO'
_tok = _secret('GITHUB_TOKEN')
_url = f'https://{_tok}@github.com/marios4371/llm_thesis.git' if _tok else 'https://github.com/marios4371/llm_thesis.git'
if not os.path.isdir(REPO_DIR):
    _r = subprocess.run(['git', 'clone', _url, REPO_DIR], capture_output=True, text=True)
    if _r.returncode:
        raise RuntimeError('git clone failed:\n' + _r.stderr.replace(_tok or '\0', '***'))
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', _url], capture_output=True)
for _args in (['fetch', 'origin', BRANCH], ['checkout', '-B', BRANCH, f'origin/{BRANCH}']):
    _r = subprocess.run(['git', '-C', REPO_DIR, *_args], capture_output=True, text=True)
    if _r.returncode:
        raise RuntimeError(f"git {' '.join(_args)} failed:\n" + _r.stderr.replace(_tok or '\0', '***'))
os.chdir(REPO_DIR)
print('code at', subprocess.run(['git', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())
print('results go to', OUT_DIR)


In [ ]:
# === Cell 2 — tests, then (first session only) the smoke run =========================
import glob, subprocess, sys

_t = subprocess.run([sys.executable, 'test_v28.py', '--require-model'], capture_output=True, text=True)
print(_t.stdout.strip().splitlines()[-1] if _t.stdout.strip() else _t.stderr[-2000:])
if _t.returncode != 0:
    print(_t.stdout[-5000:], _t.stderr[-3000:])
    raise RuntimeError('test_v28.py failed: do NOT start the GPU run')

# The smoke runs the whole GPU path on 3 rows (~10-15 min, own files, not on Drive).
# It is skipped once real results exist, i.e. from the second session on.
if not glob.glob(f'{OUT_DIR}/pretest_v28_w*.json'):
    _s = subprocess.run([sys.executable, '-u', 'pretest_v28.py', '--smoke', '--worker', '0', '--device', '0',
                         '--out-dir', '/content/v28_smoke', '--max-hours', '0.75'], capture_output=True, text=True)
    print(_s.stdout[-6000:])
    if _s.returncode != 0 or 'worker 0 finished' not in _s.stdout:
        print(_s.stderr[-6000:])
        raise RuntimeError('the smoke run failed: do NOT start the long run')
else:
    print('results already on Drive: smoke skipped, resuming')


In [ ]:
# === Cell 3 — the four steps, resumable ==============================================
# FV first on both workers (PRIMARY), then RW (ATTRIBUTE). A finished step exits in
# seconds, so every session simply runs the queue again from the top.
import subprocess, sys, time

SESSION_HOURS = 11.5    # stop starting work after this; an earlier end of the session is fine
T0 = time.time()
PARALLEL = NGPU >= 2 or GPU_GB >= 34           # two workers at once: two GPUs, or one A100
DEV = {0: '0', 1: '1' if NGPU >= 2 else '0'}

def _left():
    return SESSION_HOURS - (time.time() - T0) / 3600

def _tail(path, n=1):
    try:
        with open(path, encoding='utf-8', errors='replace') as fh:
            return ''.join(fh.readlines()[-n:]).rstrip()
    except OSError:
        return ''

def _start(w, arm):
    log = f'{OUT_DIR}/v28_worker{w}_{arm}.log'
    cmd = [sys.executable, '-u', 'pretest_v28.py', '--worker', str(w), '--device', DEV[w], '--arms', arm,
           '--out-dir', OUT_DIR, '--max-hours', f'{max(0.05, _left()):.2f}']
    return log, subprocess.Popen(cmd, stdout=open(log, 'a'), stderr=subprocess.STDOUT)

def _wait(procs):
    while any(p.poll() is None for _, p in procs):
        time.sleep(300)
        for log, p in procs:
            print(f'[{time.strftime("%H:%M")}] {log.rsplit("/", 1)[-1]}: {_tail(log)}', flush=True)
    for log, p in procs:
        print(f'\n===== {log.rsplit("/", 1)[-1]} (exit {p.returncode}) =====\n' + _tail(log, 15), flush=True)

print('two workers at once' if PARALLEL else 'one worker at a time (one GPU)')
for arm in ('FV', 'RW'):
    steps = [(0, arm), (1, arm)]
    if PARALLEL:
        if _left() < 0.25:
            break
        _wait([_start(w, a) for w, a in steps])
    else:
        for w, a in steps:
            if _left() < 0.25:
                break
            print(f'\n>>> worker {w}, arm {a}  ({_left():.1f} h left in this session)', flush=True)
            _wait([_start(w, a)])
print(f'\nsession done after {(time.time() - T0) / 3600:.1f} h')


In [ ]:
# === Cell 4 — the pre-registered reading (no GPU) ====================================
# Until every step has finished this prints PARTIAL; run all the cells again in a new session.
!python -u pretest_v28.py --summary-only --out-dir "$OUT_DIR"
